In [ ]:
# papermill parameters

# i/o
input_ohlcv_file='stables_1d.parquet'
input_gjr_file='garch.parquet'

output_feat_file='chronos-2-features.parquet'
output_pred_file='pred.parquet'
output_eval_file='chronos-2-eval.parquet'

# chronos-2
device_map = 'cpu'
cross_learning = True
lookback = 60
lookforward = 1
norm_axis = 'rows' # rows: time series, columns: cross-sectional

reference_pair = 'BTCUSDT'

# derived, forecasted metrics
gjr_garch_dist = 'skewt'

In [ ]:
print(f'''
input_ohlcv_file = {input_ohlcv_file}
input_gjr_file = {input_gjr_file}

output_feat_file = {output_feat_file}
output_pred_file = {output_pred_file}
output_eval_file = {output_eval_file}

device_map = {device_map}
cross_learning = {cross_learning}
lookback = {lookback}
lookforward = {lookforward}
norm_axis = {norm_axis}

reference_pair = {reference_pair}

# derived, forecasted metrics
gjr_garch_dist = {gjr_garch_dist}
''')

if lookforward != 1:
    raise ValueError("BUG: code can't handle lookforwards other than 1")

In [ ]:
# volatility, taker buy/sell ratio, momentum, auto correlation, level

import polars as pl
from arch import arch_model
import numpy as np
from tqdm import tqdm

ohlcv = pl.read_parquet(input_ohlcv_file).sort(['symbol', 'ts'])
gjr = pl.read_parquet(input_gjr_file).sort(['symbol', 'ts'])

df = (ohlcv
    .join(gjr, on=['symbol','ts'])
    .filter(
        (pl.col('open') > 0) & (pl.col('high') > 0) &
        (pl.col('low') > 0) & (pl.col('close') > 0) &
        (pl.col('base_volume') > 0) & 
        (pl.col('quote_volume') > 0))
    .with_columns([
        # todays log return (target)
        (pl.col('close') / pl.col('close').shift(1))
            .log()
            .over('symbol')
            .alias('ret')])
    .select([
        # base columns
        pl.col('ts'),
        pl.col('symbol'),
        pl.col('ret'),

        # log wick symmetry
        ((pl.col('high') - pl.col('close') + pl.lit(1e-6)) /
         (pl.col('close') - pl.col('low') + pl.lit(1e-6)))
            .log()
            .over('symbol')
            .alias('sym'),

        pl.col('quote_volume'),
    
        # rogers-satchell volatility
        (((pl.col('high') / pl.col('close')).log() * (pl.col('high') / pl.col('open')).log()) +
         ((pl.col('low') / pl.col('close')).log() * (pl.col('low') / pl.col('open')).log()))
            .sqrt()
            .alias('sigma_rs'),

        (pl.col('quote_volume') / pl.col('base_volume')).alias('vwap'),

        # cumulative volume delta
        (pl.col('taker_buy_base_volume') - pl.col('taker_buy_quote_volume')).alias('cvd'),
    
        # n-day momentum
        (pl.col('ret').rolling_sum(window_size=120)
            .over('symbol')
            .alias('m120')),
        (pl.col('ret').rolling_sum(window_size=30)
            .over('symbol')
            .alias('m30')),
    
        # gjr-garch
        pl.col('forecast').alias('sigma_forecast'),
        pl.col('mu'), pl.col('omega'), pl.col('alpha[1]'), pl.col('gamma[1]'),
        pl.col('beta[1]'), pl.col('eta'), pl.col('lambda'), 
    ])
    .drop_nulls()
    .write_parquet(output_feat_file))

In [ ]:
# inference with chronos-2

import numpy as np
import pandas as pd
from chronos import Chronos2Pipeline
import datetime as dt
from tqdm import tqdm

pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=device_map)
pred = []
df = pl.read_parquet(output_feat_file)

for td in tqdm(df['ts'].unique().sort()):
    lb = td - dt.timedelta(days=lookback)

    # extract batch
    win = (df
        .filter((pl.col('ts') >= lb) & (pl.col('ts') <= td))
        .drop_nulls()
        .upsample(time_column="ts", every="1d", group_by="symbol")
        .with_columns(pl.all().forward_fill())
        .filter(
            (pl.len().over("symbol") >= lookback) & (pl.col("ts").max().over("symbol") == td))
    )

    if len(win) < 3:
        continue

    future_df = pl.DataFrame()
    
    # forecast volatility
    for sym in win['symbol'].unique():
        row = df.filter((pl.col('ts') == td) & (pl.col('symbol') == sym))
        if row.height != 1:
            continue
        if row['sigma_forecast'].is_nan().all():
            var = np.full(lookforward, np.nan)
        else:
            params = np.array([
                row['mu'][0],
                row['omega'][0],
                row['alpha[1]'][0],
                row['gamma[1]'][0],
                row['beta[1]'][0],
                row['eta'][0],
                row['lambda'][0]
            ])
            ret = win.filter(pl.col('symbol') == sym).select('ret').to_numpy().flatten()
            fc = arch_model(
                    ret * 100,
                    vol='Garch',
                    p=1,
                    o=1,
                    q=1,
                    dist=gjr_garch_dist,
                    rescale=False
                ).forecast(params=params, horizon=lookforward, reindex=False)
            var = np.sqrt(fc.variance.values[-1, :]) / 100

        future_df = pl.concat([future_df, pl.DataFrame({
            'ts': [td + dt.timedelta(days=d+1) for d in range(lookforward)],
            'sigma_forecast': var,
            'symbol': sym
        })])

    # remove garch parameters
    win = win.drop(['mu','omega','alpha[1]','gamma[1]','beta[1]','eta','lambda'])

    # normalize batch
    features = [c for c in win.columns if c not in ['ts', 'symbol', 'ret']]
    if norm_axis == 'rows':
        norm = win.with_columns([
            ((pl.col(c) - pl.col(c).mean().over('symbol')) /
             (pl.col(c).std().over('symbol') + 1e-9)).alias(c)
            for c in features])
        future_norm = future_df.with_columns([
            ((pl.col(c) - pl.col(c).mean().over('symbol')) /
             (pl.col(c).std().over('symbol') + 1e-9)).alias(c)
            for c in ['sigma_forecast']])
    elif norm_axis == 'columns':
        norm = win.with_columns([
            ((pl.col(c) - pl.col(c).mean().over("ts")) / 
             (pl.col(c).std().over("ts") + 1e-9)).alias(c)
            for c in features])
        future_norm = win.with_columns([
            ((pl.col(c) - pl.col(c).mean().over("ts")) / 
             (pl.col(c).std().over("ts") + 1e-9)).alias(c)
            for c in ['sigma_forecast']])
    else:
        raise Exception(f'unknown normalization direction {norm_axis}')
    
    # Generate predictions with covariates
    pdf = pipeline.predict_df(
        norm.sort(['ts','symbol']).to_pandas(),
        future_norm.sort(['ts','symbol']).to_pandas(),
        prediction_length=lookforward,
        quantile_levels=[0.25,0.5,0.75],
        id_column="symbol",
        timestamp_column="ts",
        target="ret",
        validate_inputs=True,
        cross_learning=cross_learning,
    )

    if pdf['0.5'].isna().any():
        raise Exception(f'NaN prediction for batch {td}')

    pdf = (
        pl.from_pandas(pdf).select([
            pl.col('ts'),
            pl.col('0.5').alias('pred'),
            pl.col('0.25').alias('lowpred'),
            pl.col('0.75').alias('highpred'),
        ])
    )
    pred.append(pdf)

pl.concat(pred).write_parquet(output_pred_file)

In [ ]:
# compute performance of a long-short portfolio of the top decile coins, weighted by USD volume, 1d holing time

df = (pl
    .read_parquet(output_feat_file)
    .with_columns(pl.col('ts').dt.cast_time_unit("ms"))
    
    # join chronos-2 predictions
    .join(pl.read_parquet(output_pred_file)
            .select([
                pl.col('ts').dt.cast_time_unit("ms"),
                pl.col('symbol'),
                pl.col('prediction').alias('pred')
            ]),
          on=['ts','symbol'],
          how='inner')
    
    # join closing prices
    .join(pl.read_parquet(input_ohlcv_file)
            .select([pl.col('ts'),pl.col('symbol'),pl.col('close')]),
          on=['ts','symbol'],
          how='inner')
        
    .sort(['ts','symbol'])

    # filter some extreme events like the Terra Luna
    .filter(pl.col('ret') < np.sqrt(5))

    # sort absolute predictions in decile
    .with_columns([
        pl.col('pred').abs()
            .qcut(
                10, 
                allow_duplicates=True, 
                labels=list(map(lambda n: f'{n}', range(1,11))))
            .over('ts')
            .alias('rank'),
        pl.col('quote_volume').alias('avg_usd_vol')
    ])

    # keep top decile (long-short)
    .filter(pl.col('rank') == '10')

    # weight by cross-sectional daily USD volume
    .with_columns([
        (pl.col('avg_usd_vol') / pl.col('avg_usd_vol').sum().over(['ts'])).alias('weight')])

    # remove all weights under 5%
    .filter(pl.col('weight') > 0.05)
    .with_columns([
        (pl.col('weight') / pl.col('weight').sum().over(['ts'])).alias('weight')])

    # 1d return
    .with_columns([
        pl.col('weight').alias('initial'),
        (pl.col('weight') * pl.col('pred').sign() * (pl.col('ret').exp() - 1)).alias('gross'),
    ])

    # incorporate 0.2% tx fee
    .with_columns([
        ((pl.col('initial') + (pl.col('weight') * pl.col('ret').exp())) * 0.002).alias('fee')
    ])
    
    # net pnl
    .with_columns([
        (pl.col('gross') - pl.col('fee')).alias('pnl')
    ]))

# check weights sum to 1 before continuing
wc = (df
    .group_by("ts")
    .agg(pl.col("weight").sum().alias("total_weight"))
    .filter((pl.col("total_weight") - 1.0).abs() > 1e-6))
if not wc.is_empty():
    print(f"Weight sum error at timestamps: {wc['ts'].to_list()}")

(df
    # aggregate per-coin returns to daily portfolio returns
    .sort('ts')
    .group_by('ts')
    .agg([
        (1 + pl.col('pnl').sum()).log().alias('strategy'),
        pl.col('fee').sum().alias('fee'),
        (pl.col('weight') * pl.col('pred').sign()).sum().alias('ls'),
    ])

    # join btc as benchmark
    .join(pl.read_parquet(output_feat_file)
            .filter(pl.col('symbol') == reference_pair)
            .select([pl.col('ts'),pl.col('ret').alias('ref')]),
        on='ts',
        how='inner')
    .with_columns([
        (pl.col('strategy') - pl.col('ref')).alias('perf')])

    # compute the equity curve staring with 0
    .with_columns([
        (pl.col('strategy').cum_sum()).alias('equity')])

    # write out results
    .write_parquet(output_eval_file))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import numpy as np
from datetime import datetime, timedelta
import pytz

# --- 1. Setup & Data Extraction ---
malaysia_tz = pytz.timezone("Asia/Kuala_Lumpur")
latest_ts = df['ts'].max()
start_ts = latest_ts - timedelta(days=lookback)

# Extract and sort by weight descending
portfolio = df.filter(pl.col('ts') == latest_ts).sort('weight', descending=True).to_pandas()
top_symbols = portfolio['symbol'].tolist()

context_data = (
    pl.read_parquet(output_feat_file)
    .filter((pl.col('symbol').is_in(top_symbols[:5])) & (pl.col('ts') >= start_ts) & (pl.col('ts') <= latest_ts))
    .with_columns([
        pl.col('ts').dt.replace_time_zone("UTC").dt.convert_time_zone("Asia/Kuala_Lumpur").alias('ts_local'),
        (pl.col('ret').cum_sum().over('symbol')).alias('cum_ret')
    ])
    .to_pandas()
)

# --- 2. Visualization Setup ---
# Taller format to accommodate the expanded table and cleaner layout
fig = plt.figure(figsize=(22, 18)) 
gs = fig.add_gridspec(3, 1, height_ratios=[1.2, 0.2, 1.2]) 
sns.set_style("darkgrid")

# --- A. Market Context: History + Dashed Forecast ---
ax1 = fig.add_subplot(gs[0, 0])
palette = sns.color_palette("husl", n_colors=len(top_symbols[:5]))

# Historical Lines
sns.lineplot(data=context_data, x='ts_local', y='cum_ret', hue='symbol', 
             ax=ax1, palette=palette, lw=3)

# Projection Logic
last_ts = context_data['ts_local'].max()
next_ts = last_ts + timedelta(days=1)

for i, symbol in enumerate(top_symbols[:5]):
    symbol_data = context_data[context_data['symbol'] == symbol]
    if symbol_data.empty: continue
    
    last_cum_ret = symbol_data['cum_ret'].iloc[-1]
    forecast_ret = portfolio[portfolio['symbol'] == symbol]['pred'].values[0]
    projected_cum_ret = last_cum_ret + forecast_ret # Projection
    
    # Plot dashed projection matching the line color
    ax1.plot([last_ts, next_ts], [last_cum_ret, projected_cum_ret], 
             color=palette[i], linestyle='--', lw=3, alpha=0.8)
    ax1.scatter(next_ts, projected_cum_ret, color=palette[i], s=60, zorder=5)

ax1.set_title(f"Market Context: {lookback}-Day History & 1-Day Forecast Projection", fontsize=22, fontweight='bold')
ax1.set_ylabel("Cumulative Return", fontsize=16)
ax1.legend(title="Top 5 Assets", title_fontsize='13', fontsize='12', loc='upper left')

# --- B. Stacked Horizontal Allocation Bar ---
ax2 = fig.add_subplot(gs[1, 0])
ax2.axis('off')

# Calculate cumulative widths for stacking
left = 0
colors = sns.color_palette("rocket_r", n_colors=len(portfolio))

for i, row in portfolio.iterrows():
    width = row['weight']
    ax2.barh(0, width, left=left, color=colors[i], edgecolor='white', height=0.5)
    # Add label if width is significant enough to see
    if width > 0.04:
        ax2.text(left + width/2, 0, row['symbol'], va='center', ha='center', 
                 color='white', fontweight='bold', fontsize=14)
    left += width

ax2.set_title("Portfolio Weight Allocation", fontsize=18, fontweight='bold', pad=10)
ax2.set_ylim(-0.5, 0.5)

# --- C. Detailed Execution Table ---
ax3 = fig.add_subplot(gs[2, 0])
ax3.axis('off')

table_display = portfolio[['symbol', 'weight', 'pred', 'close', 'avg_usd_vol']].copy()
raw_preds = table_display['pred'].values 

# Formatting
table_display['weight'] = table_display['weight'].map('{:.2%}'.format)
table_display['pred'] = table_display['pred'].map('{:.4%}'.format) 
table_display['close'] = table_display['close'].map('${:,.4f}'.format)
table_display['avg_usd_vol'] = table_display['avg_usd_vol'].map('${:,.0f}'.format)
table_display.columns = ['Asset Pair', 'Target Weight', 'Forecasted Return', 'Expected Entry (Close)', '24h USD Volume']

tbl = ax3.table(cellText=table_display.values, colLabels=table_display.columns, 
                loc='center', cellLoc='center')

tbl.auto_set_font_size(False)
tbl.set_fontsize(16)
tbl.scale(1, 4.0) # Tall rows for readability

# Row-specific styling (desaturation for < 0.01% returns)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold', color='white', fontsize=18)
        cell.set_facecolor('#34495e')
    else:
        # Threshold 0.0001 (0.01%)
        if raw_preds[row-1] < 0.0001:
            cell.get_text().set_color('#bdc3c7') # Muted silver
            cell.get_text().set_style('italic')
        else:
            cell.get_text().set_color('#2c3e50')
        cell.set_facecolor('#ffffff')

# --- Header ---
current_time_myt = datetime.now(malaysia_tz)
plt.suptitle(f"Portfolio Deployment Strategy | {current_time_myt.strftime('%A, %d %B %Y | %H:%M')} MYT", 
             fontsize=26, fontweight='bold', y=0.96)

plt.tight_layout(rect=[0, 0.05, 1, 0.93])
plt.show()

In [ ]:
from IPython.display import HTML

# 1. Generate the Links List
links_html = "<h4>Binance Trade Links</h4><ul>"
for i, row in portfolio.iterrows():
    # Clean the symbol for the URL (Binance typically uses BTC_USDT or BTCUSDT)
    # Most Binance URLs use the base symbol + _USDT
    base_symbol = str(row['symbol']).replace('USDT', '').replace('/', '')
    url = f"https://demo.binance.com/trade/{base_symbol}_USDT?type=spot"
    links_html += f'<li><a href="{url}" target="_blank">{base_symbol} Trade Page</a></li>'
links_html += "</ul><hr>"
display(HTML(links_html))

# Start of the plain table string
html_table = """
<table>
  <thead>
    <tr>
      <th>Asset</th>
      <th>Weight</th>
      <th>Forecast</th>
      <th>Target Entry</th>
    </tr>
  </thead>
  <tbody>
"""

# Fill rows from the 'portfolio' DataFrame
for i, row in portfolio.iterrows():
    # Remove 'USDT' from the symbol name for a cleaner look
    clean_symbol = str(row['symbol']).replace('USDT', '').replace('/', '')
    
    html_table += f"""
    <tr>
      <td>{clean_symbol}</td>
      <td>{row['weight']:.2%}</td>
      <td>{row['pred']:.4%}</td>
      <td>{row['close']:,.8f}</td>
    </tr>
    """

html_table += "</tbody></table>"

# This will render the table in your Jupyter output area
HTML(html_table)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

res = pd.read_parquet(output_eval_file)
res.index = pd.to_datetime(res.ts)

fig, [ax1, ax2, ax3, ax4] = plt.subplots(4,1,figsize=(14,10), sharex=True)

# returns
((np.exp(res.strategy) - 1) * 100).rolling(30,min_periods=1).mean().plot(y='strategy',ax=ax1,color='gray',label='strategy')
((np.exp(res.perf) - 1) * 100).rolling(30,min_periods=1).mean().plot(y='perf',ax=ax1,color='black',label='alpha vs btc')
ax1.set_title('daily returns (30d sma)')
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(symbol='%'))
ax1.axhline(0, color='red', linestyle='--', alpha=0.5)
ax1.legend()

# equity multiplier
np.exp(res.equity).plot(y='equity',ax=ax2,color='blue',title='equity growth')
ax2.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:.2f}x'))
ax2.axhline(1, color='gray', linestyle='--', alpha=0.5)
ax2.set_yscale('log')

# fees
(res.fee * 10_000).rolling(10,min_periods=1).mean().plot(y='fee',ax=ax3,title='fees (10d sma)')
ax3.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:.0f}bps'))

# long/short ratio
res.ls.rolling(10,min_periods=1).mean().plot(y='long/short',ax=ax4,title='long/short ratio (10d sma)')
ax4.axhline(0, color='black', linestyle='--', alpha=0.5)

#ax3.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:.0f}bps'))

plt.tight_layout()
plt.show()

In [ ]:
import scrapbook as sb
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

res = pl.read_parquet(output_eval_file)

# 1. Prepare Yearly Metrics
# We use the 'strategy' (log ret) and 'perf' (log excess ret) columns
yearly_stats = (
    res.with_columns(pl.col('ts').dt.year().alias('year'))
    .group_by('year')
    .agg([
        # Annualized IR: (Mean Excess / Std Excess) * sqrt(365)
        ((pl.col('perf').mean() / (pl.col('perf').std() + 1e-9)) * np.sqrt(365)).alias('ir'),
        
        # Max Drawdown: Min of (Equity / Running Max - 1)
        ((pl.col('strategy').cum_sum().exp() / 
          pl.col('strategy').cum_sum().exp().cum_max()) - 1).min().alias('mdd'),
          
        # Growth Multiplier: exp(sum of log returns)
        (pl.col('strategy').sum().exp()).alias('growth')
    ])
    .sort('year')
)

# 2. Calculate Total Period Metrics
total_ir = (res['perf'].mean() / (res['perf'].std() + 1e-9)) * np.sqrt(365)
total_equity = res['strategy'].cum_sum().exp()
total_mdd = ((total_equity / total_equity.cum_max()) - 1).min()
total_growth = total_equity.tail(1).item()

# 3. Record Metrics for Papermill (using Scrapbook)
sb.glue("ir_total", float(total_ir))
sb.glue("mdd_total", float(total_mdd))
sb.glue("growth_total", float(total_growth))
sb.glue("yearly_stats", yearly_stats.to_pandas().to_dict(orient='records'))

# 4. Yearly Bar Chart Visualization
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
y_pd = yearly_stats.to_pandas()
years = y_pd['year'].astype(str)

# IR Chart
axes[0].bar(years, y_pd['ir'], color='skyblue', edgecolor='black')
axes[0].set_title(f'Information Ratio (Total: {total_ir:.2f})')
axes[0].axhline(0, color='black', lw=1)

# MDD Chart
axes[1].bar(years, y_pd['mdd'] * 100, color='salmon', edgecolor='black')
axes[1].set_title(f'Max Drawdown (Total: {total_mdd*100:.1f}%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

# Growth Multiplier
axes[2].bar(years, y_pd['growth'], color='lightgreen', edgecolor='black')
axes[2].set_title(f'Yearly Growth (Total: {total_growth:.2f}x)')
axes[2].yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:.2f}x'))
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()